In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.2 MB/s eta 0:00:00


In [ ]:
# !unzip /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_Sheets_TestOnly.zip

Archive:  /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_Sheets_TestOnly.zip
   creating: content/OMR_5Fold_Sheets_TestOnly/
   creating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/
   creating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/
   creating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_57_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_59_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_60_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_22_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_20_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_2_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_6_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_37_1.png  
  inflating: content/OMR_5Fold_Sheets_Tes

In [ ]:
# !unzip /content/drive/MyDrive/OMR-Datasets/ModelAnswer.zip -d /content/drive/MyDrive/OMR-Datasets/train_detect/

Archive:  /content/drive/MyDrive/OMR-Datasets/ModelAnswer.zip
   creating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/
   creating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/exam0/
  inflating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/exam0/modelAnswer_exam0_1.png  
  inflating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/exam0/modelAnswer_exam0_2.png  
  inflating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/exam0/modelAnswer_exam0_3.png  
   creating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/exam1/
  inflating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/exam1/modelAnswer_exam1_1.png  
   creating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/exam2/
  inflating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/exam2/modelAnswer_exam2_1.png  
   creating: /content/drive/MyDrive/OMR-Datasets/train_detect/ModelAnswer/exam3/
  inflati

In [ ]:
!unzip exam6-camera.zip

Archive:  exam6-camera.zip
   creating: exam6-camera/
  inflating: exam6-camera/exam6_01.jpg  
  inflating: exam6-camera/exam6_02.jpg  
  inflating: exam6-camera/exam6_03.jpg  
  inflating: exam6-camera/exam6_04.jpg  
  inflating: exam6-camera/exam6_05.jpg  
  inflating: exam6-camera/exam6_06.jpg  
  inflating: exam6-camera/exam6_07.jpg  
  inflating: exam6-camera/exam6_08.jpg  
  inflating: exam6-camera/exam6_09.jpg  
  inflating: exam6-camera/exam6_10.jpg  
  inflating: exam6-camera/exam6_11.jpg  
  inflating: exam6-camera/exam6_12.jpg  
  inflating: exam6-camera/exam6_13.jpg  
  inflating: exam6-camera/exam6_14.jpg  
  inflating: exam6-camera/exam6_15.jpg  
  inflating: exam6-camera/exam6_16.jpg  
  inflating: exam6-camera/exam6_17.jpg  
  inflating: exam6-camera/exam6_18.jpg  
  inflating: exam6-camera/exam6_19.jpg  
  inflating: exam6-camera/exam6_20.jpg  
  inflating: exam6-camera/exam6_21.jpg  
  inflating: exam6-camera/exam6_22.jpg  
  inflating: exam6-camera/exam6_23.jpg  
  i

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from ultralytics import YOLO
import json

# =============================================================================
# --- 1. CẤU HÌNH ĐƯỜNG DẪN 5 FOLDS & PIPELINE ---
# =============================================================================

TEMPLATES_JSON_FILE = '/content/drive/MyDrive/OMR-Datasets/eval_on_new_data/model_answer/templates_info.json'
GROUND_TRUTH_JSON_FILE = '/content/drive/MyDrive/OMR-Datasets/eval_on_new_data/model_answer/ground_truth.json'

REPORT_OUTPUT_FILE = '/content/omr_pipeline_5fold_detailed_eff_2.csv'
SUMMARY_OUTPUT_FILE = '/content/omr_pipeline_5fold_summary_eff_2.csv'

# Thư mục gốc chứa 5 Fold (Dành cho ảnh bài làm của học sinh)
BASE_FOLDS_DIR = '/content/exam6'

# Model Detect chung để tìm layout trên ảnh mẫu (Dùng chung cho cả 5 Fold)
YOLO_DETECT_PATH = '/content/drive/MyDrive/OMR-Datasets/train_detect/Yolo26s/OMR_Localization/yolo26s_omr_detect/weights/best.pt'
WEIGHT_DIR_CLS = '/content/drive/MyDrive/OMR-Datasets/train-cls-v2/scene2/EfficientNetB0-v2'  # <===== Đổi kịch bản tại đây
TEMPLATE_PATH = '/content/drive/MyDrive/OMR-Datasets/eval_on_new_data/model_answer'

MODEL_TYPE = 'efficientnet'  # 'yolo' hoặc 'efficientnet'
TARGET_EXAM_ID = 'exam_6'
GT_CODE_MAP = {1: 'confirmed', 2: 'crossedout', 3: 'empty'}
NUM_CLASSES = 3
CLASS_NAMES = ['confirmed', 'crossedout', 'empty']
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transform cho EfficientNet (Đã đồng bộ chuẩn 128x128)
eff_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# =============================================================================
# --- 2. CÁC HÀM XỬ LÝ (GIỮ NGUYÊN 100% TỪ CODE GỐC CỦA BẠN) ---
# =============================================================================
def clean_mat_string(numpy_element):
    try:
        while isinstance(numpy_element, np.ndarray) and numpy_element.size > 0:
            numpy_element = numpy_element[0]
        return str(numpy_element).strip()
    except:
        return ""

def pad_to_square_cv2(img, fill_color=(255, 255, 255)):
    """Hàm đắp viền trắng chống méo ảnh (Tích hợp để đồng bộ với Train)"""
    h, w = img.shape[:2]
    max_wh = max(w, h)
    top = (max_wh - h) // 2
    bottom = max_wh - h - top
    left = (max_wh - w) // 2
    right = max_wh - w - left
    return cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=fill_color)

def load_efficientnet_model(model_path):
    print(f"Loading EfficientNet-B0 from {model_path}...")
    model = models.efficientnet_b0(weights=None)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, NUM_CLASSES)
    try:
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    except Exception as e:
        print(f"❌ Lỗi load file .pth: {e}")
        return None
    model.to(DEVICE)
    model.eval()
    return model

def predict_batch_efficientnet(model, roi_imgs_list):
    if not roi_imgs_list: return []
    batch_tensors = []
    for img in roi_imgs_list:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = eff_transforms(img_rgb)
        batch_tensors.append(tensor)
    input_batch = torch.stack(batch_tensors).to(DEVICE)
    with torch.no_grad():
        outputs = model(input_batch)
        _, preds = torch.max(outputs, 1)
    results = [CLASS_NAMES[idx] for idx in preds.cpu().numpy()]
    return results

def load_templates_json(json_path):
    print(f"Loading Model Metadata from {json_path}...")
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Lỗi đọc file JSON template: {e}")
        return {}

    all_models = {}
    for exam_id, info in data.items():
        structure = info['structure']
        question_configs = []

        # Tự động sinh danh sách câu hỏi dựa vào mảng structure
        for q_idx, num_opts in enumerate(structure):
            q_name = f"Q{q_idx+1}"
            valid_options = [chr(65+k) for k in range(num_opts)] # Sinh A, B, C, D...
            question_configs.append({
                'q_name': q_name,
                'options': valid_options,
                'weight': 1.0
            })

        # Lưu vào dict (Giả định đề thi mới chỉ có 1 trang/page_num = 1)
        all_models[exam_id] = {
            1: {
                'question_configs': question_configs,
                'image_name': info['image_name']
            }
        }

    print(f"✅ Đã tải cấu hình cho các Exam ID: {list(all_models.keys())}")
    return all_models

def align_images(im1, im2, max_features=5000, keep_percent=0.2):
    img1Gray = cv2.cvtColor(im1, cv2.COLOR_BGR2GRAY)
    img2Gray = cv2.cvtColor(im2, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(max_features)
    keypoints1, descriptors1 = orb.detectAndCompute(img1Gray, None)
    keypoints2, descriptors2 = orb.detectAndCompute(img2Gray, None)
    if descriptors1 is None or descriptors2 is None: return None
    matcher = cv2.DescriptorMatcher_create(cv2.DESCRIPTOR_MATCHER_BRUTEFORCE_HAMMING)
    matches = matcher.match(descriptors1, descriptors2, None)
    matches = sorted(matches, key=lambda x: x.distance)
    keep = int(len(matches) * keep_percent)
    matches = matches[:keep]
    points1 = np.zeros((len(matches), 2), dtype=np.float32)
    points2 = np.zeros((len(matches), 2), dtype=np.float32)
    for i, match in enumerate(matches):
        points1[i, :] = keypoints1[match.queryIdx].pt
        points2[i, :] = keypoints2[match.trainIdx].pt
    try:
        h, mask = cv2.findHomography(points1, points2, cv2.RANSAC)
        if h is None: return None
        height, width, channels = im2.shape
        aligned_img = cv2.warpPerspective(im1, h, (width, height))
        return aligned_img
    except Exception as e: return None

def sort_contours_grid(rects):
    if not rects: return []
    heights = [r[3] for r in rects]
    avg_height = np.mean(heights)
    threshold_y = avg_height * 0.6
    rects = sorted(rects, key=lambda b: b[1])
    sorted_rects = []
    current_row = []
    last_y = rects[0][1]
    for rect in rects:
        y = rect[1]
        if abs(y - last_y) <= threshold_y:
            current_row.append(rect)
        else:
            current_row.sort(key=lambda b: b[0])
            sorted_rects.extend(current_row)
            current_row = [rect]
            last_y = y
    if current_row:
        current_row.sort(key=lambda b: b[0])
        sorted_rects.extend(current_row)
    return sorted_rects

def get_template_layout(template_path, detect_model, questions_structure):
    print(f"--- Phân tích ảnh mẫu: {os.path.basename(template_path)} ---\n")
    img = cv2.imread(template_path)
    if img is None: raise ValueError("Không tìm thấy ảnh mẫu!")
    total_bubbles_needed = sum(questions_structure)
    print(f"   > Mục tiêu: Cần tìm {total_bubbles_needed} ô.")
    try_confs = [0.5, 0.4, 0.3, 0.25, 0.15]
    found_rects = []
    for conf in try_confs:
        results = detect_model(img, conf=conf, verbose=False)
        raw_rects = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            w, h = x2 - x1, y2 - y1
            raw_rects.append([int(x1), int(y1), int(w), int(h)])
        if len(raw_rects) == 0: continue

        areas = [r[2] * r[3] for r in raw_rects]
        median_area = np.median(areas)
        step1_rects = []
        for r in raw_rects:
            area = r[2] * r[3]
            if area > 0.4 * median_area and area < 2.5 * median_area:
                step1_rects.append(r)

        def is_containing(boxA, boxB):
            xa, ya, wa, ha = boxA
            xb, yb, wb, hb = boxB
            xa2, ya2 = xa + wa, ya + ha
            xb2, yb2 = xb + wb, yb + hb
            padding = 5
            return (xa < xb + padding) and (ya < yb + padding) and \
                   (xa2 > xb2 - padding) and (ya2 > yb2 - padding)

        indices_to_remove = set()
        N = len(step1_rects)
        for i in range(N):
            for j in range(N):
                if i == j: continue
                if is_containing(step1_rects[i], step1_rects[j]):
                    indices_to_remove.add(i)
                    break

        final_rects = []
        for i in range(N):
            if i not in indices_to_remove:
                final_rects.append(step1_rects[i])
        print(f"   > Conf={conf}: Raw={len(raw_rects)} -> FilterArea={len(step1_rects)} -> FilterContainer={len(final_rects)}")
        found_rects = final_rects
        if len(found_rects) == total_bubbles_needed:
            print("   ✅ Đã tìm thấy ĐỦ số lượng ô!")
            break

    if len(found_rects) != total_bubbles_needed:
        if len(found_rects) > total_bubbles_needed:
             print("   ⚠ Vẫn thừa ô. Fallback: Lấy N ô có diện tích nhỏ nhất gần với Median.")
             areas = [r[2]*r[3] for r in found_rects]
             median_final = np.median(areas)
             found_rects.sort(key=lambda r: abs((r[2]*r[3]) - median_final))
             found_rects = found_rects[:total_bubbles_needed]
        else:
            raise ValueError(f"❌ Lỗi Detect: Cần {total_bubbles_needed}, tìm thấy {len(found_rects)}.")

    sorted_rects = sort_contours_grid(found_rects)
    layout_map = []
    current_idx = 0
    for q_idx, num_opts in enumerate(questions_structure):
        q_num = q_idx + 1
        options = [chr(65+k) for k in range(num_opts)]
        for opt in options:
            rect = sorted_rects[current_idx]
            layout_map.append({
                'label': f"Q{q_num}_{opt}",
                'rect': rect
            })
            current_idx += 1
    return layout_map, img

def evaluate_recognition_performance(student_answers, gt_answer_types, question_configs, roi_labels, roi_indices):
    question_stats = {'total_questions': 0, 'correct_questions': 0, 'wrong_details': []}
    label_to_gt_idx = {label: idx for label, idx in zip(roi_labels, roi_indices)}
    for q_config in question_configs:
        q_name = q_config['q_name']
        options = q_config['options']
        gt_choices = set()
        for opt in options:
            key = f"{q_name}_{opt}"
            if key in label_to_gt_idx:
                gt_idx = label_to_gt_idx[key]
                if gt_idx < len(gt_answer_types):
                    gt_code = gt_answer_types[gt_idx]
                    if gt_code == 1: gt_choices.add(opt)
        pred_choices = set()
        for opt in options:
            key = f"{q_name}_{opt}"
            if student_answers.get(key) == 'confirmed': pred_choices.add(opt)
        question_stats['total_questions'] += 1
        if gt_choices == pred_choices:
            question_stats['correct_questions'] += 1
        else:
            gt_str = ",".join(sorted(gt_choices)) if gt_choices else "Empty"
            pred_str = ",".join(sorted(pred_choices)) if pred_choices else "Empty"
            question_stats['wrong_details'].append(f"{q_name}: GT=[{gt_str}] vs AI=[{pred_str}]")
    return question_stats

# =============================================================================
# --- 3. VÒNG LẶP CHÍNH QUA 5 FOLDS ---
# =============================================================================
def main():
    print("--- PIPELINE ĐÁNH GIÁ 5-FOLD CHẤM BÀI HỌC SINH TỰ ĐỘNG (ALIGN & DETECT & CLS) ---\n")

    # 1. Load Model Detect (Chỉ cần Load 1 lần dùng chung cho mọi Fold vì Template không đổi)
    print("Loading YOLO Detect model...")
    try:
        detect_model = YOLO(YOLO_DETECT_PATH)
    except:
        print("⚠ Lỗi tải model detect. Kiểm tra lại đường dẫn.")
        return

    # 2. Tải Metadata cấu trúc đề thi
    # Tải Metadata cấu trúc đề thi từ JSON
    all_models = load_templates_json(TEMPLATES_JSON_FILE)
    if TARGET_EXAM_ID is not None and TARGET_EXAM_ID not in all_models:
        print(f"❌ Không tìm thấy dữ liệu cho Exam {TARGET_EXAM_ID}")
        return

    print(f"Loading Student Exams from {GROUND_TRUTH_JSON_FILE}...")
    try:
        with open(GROUND_TRUTH_JSON_FILE, 'r', encoding='utf-8') as f:
            exams_data = json.load(f) # Nó sẽ là một Dictionary dạng {"anh_1.png": {...}, "anh_2.png": {...}}
    except Exception as e:
        print(f"Lỗi mở file ground truth JSON: {e}")
        return
    # ====================================================================
    # BƯỚC PRE-PROCESSING TẠO LAYOUT (Chỉ chạy 1 lần)
    # ====================================================================
    exam_target_str = TARGET_EXAM_ID if TARGET_EXAM_ID is not None else 'ALL'
    print(f"\n--- Đang khởi tạo Template Layout (Detect) cho Exam: {exam_target_str} ---")
    exam_templates_cache = {}
    exams_to_process = [TARGET_EXAM_ID] if TARGET_EXAM_ID is not None else list(all_models.keys())

    for ex_id in exams_to_process:
        print(f"\n>> Khởi tạo Template cho Exam {ex_id}:")
        exam_pages = all_models[ex_id]
        for page_num, page_info in exam_pages.items():
            tpl_image_name = page_info['image_name']
            tpl_path = os.path.join(TEMPLATE_PATH, f"exam{ex_id}", tpl_image_name)
            if not os.path.exists(tpl_path):
                 tpl_path = os.path.join(TEMPLATE_PATH, tpl_image_name)
            if not os.path.exists(tpl_path):
                print(f"⚠ Không tìm thấy ảnh mẫu: {tpl_path}")
                continue

            q_configs = page_info['question_configs']
            structure_list = [len(q['options']) for q in q_configs]

            try:
                layout_map, template_img = get_template_layout(tpl_path, detect_model, structure_list)
                exam_templates_cache[(ex_id, page_num)] = (layout_map, template_img)
            except Exception as e:
                print(f"     ❌ Lỗi tạo layout Exam {ex_id} Page {page_num}: {e}")

    # ====================================================================
    # BƯỚC ĐÁNH GIÁ 5 FOLD CROSS VALIDATION
    # ====================================================================
    fold_results = []
    all_detailed_reports = []

    for fold in range(1, 6):
        print(f"\n{'='*60}")
        print(f"🔍 ĐANG ĐÁNH GIÁ FOLD {fold} / 5")
        print(f"{'='*60}")

        fold_dir = os.path.join(BASE_FOLDS_DIR, f"Fold_{fold}")
        current_test_dir = '/content/exam6-camera'
        cur_weight_dir = os.path.join(WEIGHT_DIR_CLS, f"Fold_{fold}")

        # Load Model Classify Tương ứng của Fold
        classify_model = None
        if MODEL_TYPE == 'yolo':
            yolo_path = os.path.join(cur_weight_dir, "weights", "best.pt")
            if not os.path.exists(yolo_path):
                print(f"❌ Không tìm thấy YOLO model tại {yolo_path}. Bỏ qua Fold {fold}.")
                continue
            classify_model = YOLO(yolo_path)

        elif MODEL_TYPE == 'efficientnet':
            eff_path = os.path.join(cur_weight_dir, f"efficientnetb0_fold{fold}_gan.pth")
            classify_model = load_efficientnet_model(eff_path)
            if classify_model is None: continue

        fold_total_q, fold_correct_q = 0, 0
        actual_total_sheets, actual_perfect_sheets = 0, 0
        exam0_sheets = {}

        # Duyệt qua các bài thi dạng Dictionary JSON
        for image_name, sheet_info in exams_data.items():
            try:
                exam_id = sheet_info['exam_id']
                if TARGET_EXAM_ID is not None and exam_id != TARGET_EXAM_ID:
                    continue

                page_num = 1 # Mặc định bộ mới 1 trang
                gt_answer_types = sheet_info.get('gt_codes', [])

                cache_key = (exam_id, page_num)
                if cache_key not in exam_templates_cache:
                    continue

                current_layout_map, current_template_img = exam_templates_cache[cache_key]
                current_q_configs = all_models[exam_id][page_num]['question_configs']
            except Exception as e:
                continue

            # KIỂM TRA PHIẾU CÓ THUỘC FOLD HIỆN TẠI KHÔNG (Rất quan trọng)
            full_image_path = os.path.join(current_test_dir, image_name)
            if not os.path.exists(full_image_path):
                 full_image_path_root = os.path.join(current_test_dir, image_name)
                 if os.path.exists(full_image_path_root):
                     full_image_path = full_image_path_root
                 else:
                     continue

            # --- A. CĂN CHỈNH ẢNH ---
            student_img = cv2.imread(full_image_path)
            aligned_img = align_images(student_img, current_template_img)
            if aligned_img is None: continue

            # --- B. CẮT ROI TỪ LAYOUT KÈM PAD ---
            roi_imgs, roi_keys, roi_indices = [], [], []
            for idx, item in enumerate(current_layout_map):
                label = item['label']
                x, y, w, h = item['rect']
                roi = aligned_img[y:y+h, x:x+w]
                if roi.size > 0:
                    roi_padded = pad_to_square_cv2(roi)
                    roi_resized = cv2.resize(roi_padded, (128, 128))
                    roi_imgs.append(roi_resized)
                    roi_keys.append(label)
                    roi_indices.append(idx)

            if not roi_imgs: continue

            # --- C. PHÂN LOẠI ---
            student_answers = {}
            if MODEL_TYPE == 'yolo':
                preds = classify_model(roi_imgs, verbose=False)
                for idx, res in enumerate(preds):
                    student_answers[roi_keys[idx]] = res.names[res.probs.top1]
            elif MODEL_TYPE == 'efficientnet':
                preds_list = predict_batch_efficientnet(classify_model, roi_imgs)
                for idx, status in enumerate(preds_list):
                    student_answers[roi_keys[idx]] = status

            # --- D. ĐÁNH GIÁ ---
            stats = evaluate_recognition_performance(
                student_answers, gt_answer_types, current_q_configs, roi_keys, roi_indices)

            fold_total_q += stats['total_questions']
            fold_correct_q += stats['correct_questions']

            all_detailed_reports.append({
                'Fold': fold,
                'Image': image_name,
                'Exam': exam_id,
                'Page': page_num,
                'Total_Q': stats['total_questions'],
                'Correct_Q': stats['correct_questions'],
                'Wrong_Log': "; ".join(stats['wrong_details'])
            })

            # Xử lý gộp phiếu Exam 0 (Y nguyên của bạn)
            if exam_id == 0:
                try:
                    base = image_name.rsplit('.', 1)[0]
                    sheet_id = base.rsplit('_', 1)[0]
                except: sheet_id = image_name
                if sheet_id not in exam0_sheets:
                    exam0_sheets[sheet_id] = {'pages': 0, 'correct_q': 0, 'total_q': 0}
                exam0_sheets[sheet_id]['pages'] += 1
                exam0_sheets[sheet_id]['correct_q'] += stats['correct_questions']
                exam0_sheets[sheet_id]['total_q'] += stats['total_questions']
            else:
                actual_total_sheets += 1
                if stats['correct_questions'] == stats['total_questions']:
                    actual_perfect_sheets += 1

        for sheet_id, data in exam0_sheets.items():
            if data['pages'] == 3:
                actual_total_sheets += 1
                if data['correct_q'] == data['total_q']:
                    actual_perfect_sheets += 1

        q_acc = (fold_correct_q / fold_total_q * 100) if fold_total_q > 0 else 0
        s_acc = (actual_perfect_sheets / actual_total_sheets * 100) if actual_total_sheets > 0 else 0

        print(f"📊 Kết quả Fold {fold}:")
        print(f"   - Question Accuracy: {q_acc:.2f}% ({fold_correct_q}/{fold_total_q})")
        print(f"   - Sheet Accuracy:    {s_acc:.2f}% ({actual_perfect_sheets}/{actual_total_sheets})")

        fold_results.append({'Fold': fold, 'Q_Acc': q_acc, 'S_Acc': s_acc})

    # =============================================================================
    # --- 4. XUẤT 2 FILE BÁO CÁO VÀ TÍNH TRUNG BÌNH ĐỘ LỆCH CHUẨN ---
    # =============================================================================
    if len(all_detailed_reports) > 0:
        df_reports = pd.DataFrame(all_detailed_reports)
        df_reports.to_csv(REPORT_OUTPUT_FILE, index=False)
        print(f"\n✅ Đã lưu log lỗi chi tiết từng trang tại: {REPORT_OUTPUT_FILE}")

    if len(fold_results) > 0:
        df_folds = pd.DataFrame(fold_results)
        q_acc_mean = df_folds['Q_Acc'].mean()
        q_acc_std = df_folds['Q_Acc'].std()
        s_acc_mean = df_folds['S_Acc'].mean()
        s_acc_std = df_folds['S_Acc'].std()

        log_q_acc = f"Độ chính xác Câu hỏi (Q_Acc): {q_acc_mean:.2f}% ± {q_acc_std:.2f}%"
        log_s_acc = f"Độ chính xác Phiếu   (S_Acc): {s_acc_mean:.2f}% ± {s_acc_std:.2f}%"

        print("\n🏆 TỔNG KẾT 5-FOLD CROSS VALIDATION TỚI BƯỚC PIPELINE 🏆")
        print("="*50)
        print(log_q_acc)
        print(log_s_acc)
        print("="*50)

        df_folds.to_csv(SUMMARY_OUTPUT_FILE, index=False)

        with open(SUMMARY_OUTPUT_FILE, 'a', encoding='utf-8') as f:
            f.write("\n")
            f.write(log_q_acc + "\n")
            f.write(log_s_acc + "\n")

        print(f"✅ Đã lưu bảng tổng hợp từng fold VÀ kèm theo dòng log kết quả tại: {SUMMARY_OUTPUT_FILE}")
    else:
        print("❌ Không có dữ liệu. Vui lòng kiểm tra lại đường dẫn BASE_FOLDS_DIR!")

if __name__ == '__main__':
    main()

--- PIPELINE ĐÁNH GIÁ 5-FOLD CHẤM BÀI HỌC SINH TỰ ĐỘNG (ALIGN & DETECT & CLS) ---

Loading YOLO Detect model...
Loading Model Metadata from /content/drive/MyDrive/OMR-Datasets/eval_on_new_data/model_answer/templates_info.json...
✅ Đã tải cấu hình cho các Exam ID: ['exam_6']
Loading Student Exams from /content/drive/MyDrive/OMR-Datasets/eval_on_new_data/model_answer/ground_truth.json...

--- Đang khởi tạo Template Layout (Detect) cho Exam: exam_6 ---

>> Khởi tạo Template cho Exam exam_6:
--- Phân tích ảnh mẫu: model_exam6_1.jpg ---

   > Mục tiêu: Cần tìm 36 ô.
   > Conf=0.5: Raw=36 -> FilterArea=36 -> FilterContainer=36
   ✅ Đã tìm thấy ĐỦ số lượng ô!

🔍 ĐANG ĐÁNH GIÁ FOLD 1 / 5
Loading EfficientNet-B0 from /content/drive/MyDrive/OMR-Datasets/train-cls-v2/scene2/EfficientNetB0-v2/Fold_1/efficientnetb0_fold1_gan.pth...
📊 Kết quả Fold 1:
   - Question Accuracy: 68.22% (307/450)
   - Sheet Accuracy:    18.00% (9/50)

🔍 ĐANG ĐÁNH GIÁ FOLD 2 / 5
Loading EfficientNet-B0 from /content/drive/

#ĐÁNH GIÁ TRÊN ẢNH SCAN



=> with gan v1

Độ chính xác Câu hỏi (Q_Acc): 85.33% ± 16.42%

Độ chính xác Phiếu   (S_Acc): 52.40% ± 25.04%

=> with gan v2

Độ chính xác Câu hỏi (Q_Acc): 75.96% ± 13.83%

Độ chính xác Phiếu   (S_Acc): 36.00% ± 15.75%

=> no gan

Độ chính xác Câu hỏi (Q_Acc): 94.49% ± 5.11%

Độ chính xác Phiếu   (S_Acc): 79.60% ± 7.13%

#ĐÁNH GIÁ ẢNH CHỤP CAMERA


=> no gan

Độ chính xác Câu hỏi (Q_Acc): 77.69% ± 24.01%

Độ chính xác Phiếu   (S_Acc): 51.60% ± 28.96%

=> with gan v1

Độ chính xác Câu hỏi (Q_Acc): 79.42% ± 13.11%

Độ chính xác Phiếu   (S_Acc): 41.20% ± 23.52%

=> with gan v2

Độ chính xác Câu hỏi (Q_Acc): 74.09% ± 8.63%

Độ chính xác Phiếu   (S_Acc): 35.20% ± 16.16%